# Project: Did the coupon work, and who stays?

*ML Academy — starter notebook*

**Goal.** Estimate how much the coupon really reduced cancellations, separating its effect from the fact that engaged customers got it, and build a survival model that ranks customers by churn risk.

**The data.** 3,000 simulated subscribers of a streaming service: engagement (hours per week), age, whether they received a retention coupon, and how many months they stayed, with censoring for those still subscribed. The marketing team targeted engaged customers, so engagement confounds the coupon's effect. The true effect is known: the coupon cuts the cancellation rate by about 30% (a hazard ratio of 0.7).

**Uses:** Lesson 29, Lesson 33, Lesson 8, Lesson 4.

Work through the steps in order. Lines marked `# TODO` are yours to write; each step ends with a check that prints ✅ when you're right (and a hint when you're not). Cells that use a step's result will show errors until that step is done — that's expected. The solution notebook is in the same folder.

Requirements: `pip install numpy pandas scikit-learn matplotlib scipy` — the data is simulated in the first step, so no download is needed.

In [1]:
# Run this cell first: it defines check(), which tells you whether each step is right.
import numpy as np
def check(get, want, hint=''):
    try:
        got = get()
        if isinstance(want, str):
            ok = str(got) == want
        else:
            ok = np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=1e-6)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


## Step 1. Load the subscriber data

Simulated data for 3,000 subscribers of a streaming service, so we know the truth. Engagement (hours watched per week) affects both who received a retention coupon (the marketing team targeted engaged customers) and how long people stay. The coupon's true effect is to cut the cancellation rate by about 30%. Some customers are still subscribed when the data was pulled: they're censored.

In [2]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(11)
n = 3000
engagement = rng.gamma(2.0, 2.0, n)
age = rng.integers(18, 70, n)
coupon = (rng.random(n) < 1 / (1 + np.exp(-(engagement - 4) / 1.5))).astype(int)
hazard = 0.08 * np.exp(-0.25 * engagement + np.log(0.7) * coupon + 0.005 * (age - 40))
churn_time = rng.exponential(1 / hazard)
observed_for = rng.uniform(3, 24, n)
df = pd.DataFrame({"engagement": engagement.round(2), "age": age, "coupon": coupon,
                   "months": np.minimum(churn_time, observed_for).round(2), "churned": (churn_time <= observed_for).astype(int)})
print(df.shape, " coupon share", round(df.coupon.mean(), 3), " churned", int(df.churned.sum()))
df.head()

(3000, 5)  coupon share 0.466  churned 948


,engagement,age,coupon,months,churned
0,3.42,66,0,11.41,1
1,7.60,68,0,19.01,0
2,2.62,69,1,4.55,1
3,5.03,57,1,11.06,0
4,5.66,30,1,3.34,0


In [3]:
check(lambda: df.shape, [3000, 5], hint='Run the cell as given.')

✅ Correct!


## Step 2. The naive comparison

Compare the share of customers who churned within the data, with and without a coupon. Why might this exaggerate the coupon's effect?

In [4]:
naive_diff = None  # TODO: churn share with coupon minus without
print("churned with coupon minus without:", round(naive_diff, 3))
print("average engagement, coupon vs none:", df.groupby("coupon").engagement.mean().round(2).tolist())

TypeError: type NoneType doesn't define __round__ method

In [5]:
check(lambda: round(naive_diff, 3), -0.26, hint="Subtract the two groups' churn shares.")

❌ Not yet. Hint: Subtract the two groups' churn shares.


## Step 3. Check overlap with propensity scores

Model each customer's probability of getting a coupon from engagement and age. If some customers had almost no chance (or almost certainty), no method can compare them fairly.

In [6]:
from sklearn.linear_model import LogisticRegression
ps_model = LogisticRegression(max_iter=1000).fit(df[["engagement", "age"]], df.coupon)
ps = None  # TODO: the predicted probability of a coupon (column 1)
print("propensity range:", round(ps.min(), 3), "to", round(ps.max(), 3))
print("share with propensity outside [0.05, 0.95]:", round(float(np.mean((ps < 0.05) | (ps > 0.95))), 3))

AttributeError: 'NoneType' object has no attribute 'min'

In [7]:
check(lambda: round(float(ps.mean()), 3), 0.466, hint='predict_proba(...)[:, 1].')

❌ Not yet. Hint: predict_proba(...)[:, 1].


## Step 4. Kaplan–Meier curves

Estimate the survival curve for each group, respecting censoring, and read off the share still subscribed after 12 months.

In [8]:
def kaplan_meier(t, e):
    s, out = 1.0, []
    for u in np.unique(t[e == 1]):
        # TODO: multiply by (1 − churned at u ÷ at risk at u)
        out.append((u, s))
    return out
def surv_at(curve, month):
    s = 1.0
    for u, v in curve:
        if u > month: break
        s = v
    return s
km = {g: kaplan_meier(df.months[df.coupon == g].to_numpy(), df.churned[df.coupon == g].to_numpy()) for g in [0, 1]}
s12 = [round(float(surv_at(km[g], 12)), 3) for g in [0, 1]]
print("still subscribed at 12 months, no coupon vs coupon:", s12)

still subscribed at 12 months, no coupon vs coupon: [1.0, 1.0]


In [9]:
check(lambda: s12, [0.582, 0.841], hint='At each churn time, multiply by 1 − d/n.')

❌ Not yet. Hint: At each churn time, multiply by 1 − d/n.


## Step 5. A Cox model adjusting for engagement

Fit a Cox proportional hazards model with coupon, engagement and age by maximising the partial likelihood. The coupon's hazard ratio, adjusted for engagement, should be near the true 0.7.

In [10]:
from scipy.optimize import minimize
Xc = df[["coupon", "engagement", "age"]].to_numpy(float); Xc[:, 2] = (Xc[:, 2] - 40) / 10
T, E = df.months.to_numpy(), df.churned.to_numpy()
order = np.argsort(-T); Xs, Es = Xc[order], E[order]
def neg_pl(beta):
    eta = Xs @ beta
    log_risk = np.logaddexp.accumulate(eta)          # log of the sum over everyone with a longer time
    return -np.sum((eta - log_risk)[Es == 1])
fit = minimize(neg_pl, np.zeros(3), method="BFGS")
hr = None  # TODO: hazard ratios are e to the power of the coefficients
print("hazard ratios — coupon", round(hr[0], 2), " engagement (per hour)", round(hr[1], 2), " age (per 10 years)", round(hr[2], 2))
naive_fit = minimize(lambda b: neg_pl(np.r_[b, 0, 0]), np.zeros(1), method="BFGS")
print("coupon hazard ratio without adjustment:", round(float(np.exp(naive_fit.x[0])), 2))

TypeError: 'NoneType' object is not subscriptable

In [11]:
check(lambda: round(float(hr[0]), 2), 0.62, hint='np.exp(fit.x).')

❌ Not yet. Hint: np.exp(fit.x).


## Step 6. How well does the model rank customers?

Compute the concordance index: among pairs where we know who churned first, the share where the model gave the earlier churner the higher risk.

In [12]:
risk = Xc @ fit.x
ok = tot = 0
for i in np.where(E == 1)[0]:
    later = T > T[i]
    tot += later.sum(); ok += (risk[i] > risk[later]).sum()
c_index = None  # TODO: concordant pairs divided by comparable pairs
print("concordance index:", round(c_index, 3))

TypeError: type NoneType doesn't define __round__ method

In [13]:
check(lambda: round(float(c_index), 3), 0.691, hint='ok / tot.')

❌ Not yet. Hint: ok / tot.


## Go further

- Estimate the coupon's effect on 12-month retention with inverse propensity weighting (Lesson 29), trimming propensities to [0.05, 0.95]. Does it agree with the Cox model?
- Add a hidden confounder to the simulation that you don't include in the model. How far off does the hazard ratio go?
- Check proportional hazards: estimate the coupon's hazard ratio separately for months 0–6 and 6–24 (Lesson 33's Going deeper).